In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
    concat,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_contact_details_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        from pyspark.sql import functions as F
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ContactDetailId	String"
        #     ",SourceSystem	String"
        #     ",PortfolioKey	String"
        #     ",PortfolioName	String"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId	String"
        #     ",ContactDetailKey	String"
        #     ",CDI	string"
        #     ",PartyKey	String"
        #     ",ContactDetailTypeCode	string"
        #     ",ContactDetailType	string"
        #     ",ContactDetailTypeGroup string"
        #     ",FullAddress	string"
        #     ",Country	string"
        #     ",County	string"
        #     ",City	string"
        #     ",PostalCode	string"
        #     ",Street	string"
        #     ",Street1 string"
        #     ",Street2 string"
        #     ",Street3 string"
        #     ",StreetNumber	string"
        #     ",PhoneNumber1	string"
        #     ",PhoneNumber2	string"
        #     ",PhoneNumber3	string"
        #     ",EmailAddress1	string"
        #     ",EmailAddress2	string"
        #     ",IsValidFlag	boolean"
        #     ",CommunicationFlag	string"
        #     ",IsPrimaryFlag	boolean"
        #     ",CreatedBy	string"
        #     ",CreatedDate	timestamp"
        #     ",LastUpdatedDate	date"
        #     ",OriginalTelephoneCode	string"
        #     ",OriginalTelephoneType	string"
        #     ",IsActiveFlag	string"
        #     ",DoNotCallFlag	boolean"
        #     ",IsDialerActivityFlag	int"
        #     ",DialerActivityTelephoneNumber	string"
        #     ",InternalConfirmationTypeCode	int"
        #     ",InternalConfirmationType	string"
        #     ",CollectionSourceTypeCode	int"
        #     ",CollectionSourceType	string"
        #     ",DoNotSendFlag boolean"
        #     ",AddContactFlag boolean"
        #     ",SerialNumber int"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ContactDetails"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qca",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")


            logging.info("Records have been deleted successfully!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ==============================dim_customer_static====================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_id",
            "cust_cif",
            "cuad_code",
            "businessdate",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )
        # ==============================dim_customer_phones====================

        dim_customer_phones_extra_filters = None

        if dim_customer_phones_extra_filters is not None:
            dim_customer_phones_extra_filters.append(date_filter)
            dim_customer_phones_final_filters = " AND ".join(
                dim_customer_phones_extra_filters
            )
        else:
            dim_customer_phones_final_filters = date_filter

        dim_customer_phones_selected_fields = [
            "cuph_code",
            "cuph_origin_type",
            "cuph_number",
            "cuph_confirmation",
            "cuph_created",
            "cuph_active",
            "cuph_do_not_call",
            "cuph_is_dialer",
            "cuph_dialer_phone",
            "cust_code",
        ]

        dim_customer_phones = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_phones",
            dim_customer_phones_final_filters,
            dim_customer_phones_selected_fields,
        )

        # Rename 'cust_code' to 'cust_code_phones'
        dim_customer_phones = dim_customer_phones.withColumnRenamed(
            "cust_code", "cust_code_phones"
        )

        # ================== FS_CASES_SNAP ==================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = ["CUST_CODE"]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        FS_CASES_SNAP = FS_CASES_SNAP.select(FS_CASES_SNAP.CUST_CODE).distinct()

        # ==============================contact_details_qca_df_phones====================

        contact_details_qca_primary_key_fields = [
            dim_customer_static.cuad_code,
            dim_customer_static.cust_id,
            dim_customer_phones.cuph_origin_type,
            lit("qca"),
        ]

        contact_details_qca_df_phones = (
            dim_customer_static.join(
                FS_CASES_SNAP,
                (FS_CASES_SNAP["CUST_CODE"] == dim_customer_static["cust_id"]),
                "INNER",
            )
            .join(
                dim_customer_phones,
                (
                    dim_customer_static["cust_id"]
                    == dim_customer_phones["cust_code_phones"]
                ),
                "left",
            )
            .join(
                portfolio,
                (
                    portfolio["Prefix"]
                    == substring(dim_customer_static["cust_cif"], 1, 3)
                ),
                "left",
            )
            .where(dim_customer_static["cuad_code"].isNotNull())
        )

        contact_details_qca_df_phones = contact_details_qca_df_phones.withColumn(
            "ContactDetailID",
            sha2(concat_ws(";", *contact_details_qca_primary_key_fields), 256),
        ).withColumn(
            "ContactDetailKey",
            concat_ws(
                ";",
                dim_customer_static.cuad_code,
                dim_customer_static.cust_id,
                dim_customer_phones.cuph_origin_type,
            ),
        )

        contact_details_qca_df_phones = contact_details_qca_df_phones.select(
            contact_details_qca_df_phones.ContactDetailID,
            lit("qca").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            contact_details_qca_df_phones.ContactDetailKey.cast("string").alias(
                "ContactDetailKey"
            ),
            dim_customer_static.cust_cif.alias("CDI"),
            contact_details_qca_df_phones.cust_id.cast("string").alias("PartyKey"),
            contact_details_qca_df_phones.cuph_origin_type.cast("string").alias(
                "ContactDetailTypeCode"
            ),
            contact_details_qca_df_phones.cuph_origin_type.cast("string").alias(
                "ContactDetailType"
            ),
            lit("Phone").cast("String").alias("ContactDetailTypeGroup"),
            lit(None).alias("FullAddress"),
            lit(None).alias("Country"),
            lit(None).alias("County"),
            lit(None).alias("City"),
            lit(None).alias("PostalCode"),
            lit(None).alias("Street"),
            lit(None).alias("Street1"),
            lit(None).alias("Street2"),
            lit(None).alias("Street3"),
            lit(None).alias("StreetNumber"),
            contact_details_qca_df_phones.cuph_number.alias("PhoneNumber1"),
            lit(None).alias("PhoneNumber2"),
            lit(None).alias("PhoneNumber3"),
            lit(None).alias("EmailAddress1"),
            lit(None).alias("EmailAddress2"),
            contact_details_qca_df_phones.cuph_confirmation.alias("IsValidFlag"),
            lit(None).alias("CommunicationFlag"),
            lit(None).alias("IsPrimaryFlag"),
            lit(None).alias("CreatedBy"),
            contact_details_qca_df_phones.cuph_created.alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("OriginalTelephoneCode"),
            lit(None).alias("OriginalTelephoneType"),
            contact_details_qca_df_phones.cuph_active.cast("string").alias(
                "IsActiveFlag"
            ),
            contact_details_qca_df_phones.cuph_do_not_call.alias("DoNotCallFlag"),
            contact_details_qca_df_phones.cuph_is_dialer.alias("IsDialerActivityFlag"),
            contact_details_qca_df_phones.cuph_dialer_phone.alias(
                "DialerActivityTelephoneNumber"
            ),
            lit(None).alias("InternalConfirmationTypeCode"),
            lit(None).alias("InternalConfirmationType"),
            lit(None).alias("CollectionSourceTypeCode"),
            lit(None).alias("CollectionSourceType"),
            lit(None).alias("DoNotSendFlag"),
            lit(None).alias("AddContactFlag"),
            lit(None).alias("SerialNumber"),
            dim_customer_static.businessdate.alias("BusinessDate"),
        )

                # ==============================dim_customer_static====================

        # Select distinct values of cust_id and cust_cif
        dim_customer_static = dim_customer_static.select(
            "cust_id", "cust_cif", col("cuad_code").alias("cuad_code_static"), "BusinessDate"
        ).distinct()

        # ==============================dim_customer_addresses====================

        dim_customer_addresses_extra_filters = None

        if dim_customer_addresses_extra_filters is not None:
            dim_customer_addresses_extra_filters.append(date_filter)
            dim_customer_addresses_final_filters = " AND ".join(
                dim_customer_addresses_extra_filters
            )
        else:
            dim_customer_addresses_final_filters = date_filter

        dim_customer_addresses_selected_fields = [
            "CUAD_CODE",
            "CUST_CODE",
            "CUAD_ADDRESS_TYPE",
            "CUAD_City",
            "CUAD_POSTCODE",
            "CUAD_Country",
            "CUAD_REGION",
            "CUAD_STREET_1",
            "CUAD_STREET_2",
            "CUAD_STREET_3",
            "CUAD_STREET_NUM",
            "CUAD_CONFIRMATION",
            "CUAD_INTERNAL_CONFIRMATION",
            "CUAD_CREATED",
            "CUAD_LAST_UPD",
            "CUAD_ACTIVE",
            "CUAD_DO_NOT_SEND_FLAG",
            "CUAD_ADD_CONTACT_FLAG",
            "BusinessDate",
        ]

        dim_customer_addresses = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_addresses",
            dim_customer_addresses_final_filters,
            dim_customer_addresses_selected_fields,
        )
        
        src = spark.table(f"`{env_prefix}-cdh`.reference.qca_dim_customer_addresses_extra")

        # Alias ALL columns to "<col>_param"
        dim_customer_addresses_param = src.select(
            *[F.col(c).alias(f"{c}_param") for c in src.columns]
        )

        # ==============================contact_details_qca_df_address====================

        contact_details_qca_primary_key_fields_2 = [
            dim_customer_addresses.CUAD_CODE,
            dim_customer_static.cust_id,
            dim_customer_addresses.CUAD_ADDRESS_TYPE,
            lit("qca"),
        ]

        contact_details_full_address_fields = [
            dim_customer_addresses.CUAD_STREET_1,
            dim_customer_addresses.CUAD_STREET_NUM,
            dim_customer_addresses.CUAD_City,
            dim_customer_addresses.CUAD_POSTCODE,
        ]

        contact_details_qca_df_address = (
            dim_customer_static.join(
                dim_customer_addresses,
                (dim_customer_addresses["CUST_CODE"] == dim_customer_static["cust_id"]),
                "left",
            )
            .join(
                portfolio,
                (
                    portfolio["Prefix"]
                    == substring(dim_customer_static["cust_cif"], 1, 3)
                ),
                "left",
            )
            .join(
                dim_customer_addresses_param,
                (dim_customer_addresses_param["cuad_code_param"] == dim_customer_static["cuad_code_static"]) &
                (dim_customer_addresses_param["cust_code_param"] == dim_customer_static["cust_id"]),
                "left",
            )
            .where(
                (dim_customer_addresses_param["cuad_code_param"].isNotNull()) | 
                (dim_customer_addresses["CUST_CODE"].isNotNull())
            )
        )

        contact_details_qca_df_address = (
            contact_details_qca_df_address.withColumn(
                "ContactDetailID",
                sha2(concat_ws(";", *contact_details_qca_primary_key_fields_2), 256),
            )
            .withColumn(
                "ContactDetailKey",
                concat_ws(
                    ";",
                    contact_details_qca_df_address.CUAD_CODE,
                    dim_customer_static.cust_id,
                    dim_customer_addresses.CUAD_ADDRESS_TYPE,
                ),
            )
            .withColumn(
                "fulladdress",
                concat_ws(";", *contact_details_full_address_fields),
            )
            .withColumn(
                "street",
                when(
                    ~coalesce(col("cuad_street_3"), lit("N/A")).isin("N/A", "", "-"),
                    col("cuad_street_3")
                ).when(
                    ~coalesce(col("cuad_street_2"), lit("N/A")).isin("N/A", "", "-"),
                    concat(
                        col("cuad_street_1").cast("string"), 
                        lit("-"), 
                        col("cuad_street_2").cast("string")
                    )
                ).otherwise(col("cuad_street_1"))
            )
        )   

        contact_details_qca_df_address = contact_details_qca_df_address.select(
            F.col("ContactDetailID"),
            F.lit("qca").alias("SourceSystem"),
            F.coalesce(portfolio.portfolio_id_qca, F.lit(2)).cast("string").alias("PortfolioKey"),
            F.coalesce(portfolio.PortfolioName, F.lit("Core Retail")).cast("string").alias("PortfolioName"),
            F.lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            F.lit(None).alias("BatchId"),
            F.col("ContactDetailKey").cast("string").alias("ContactDetailKey"),
            F.col("cust_cif").alias("CDI"),
            F.col("cust_id").cast("string").alias("PartyKey"),
            F.coalesce(F.col("CUAD_ADDRESS_TYPE"), F.col("CUAD_ADDRESS_TYPE_param")).cast("string").alias("ContactDetailTypeCode"),
            F.coalesce(F.col("CUAD_ADDRESS_TYPE"), F.col("CUAD_ADDRESS_TYPE_param")).cast("string").alias("ContactDetailType"),
            F.lit("Address").cast("string").alias("ContactDetailTypeGroup"),
            F.concat_ws(
                ";",
                F.coalesce(F.col("CUAD_STREET_1"),  F.col("CUAD_STREET_1_param")),
                F.coalesce(F.col("CUAD_STREET_NUM"),F.col("CUAD_STREET_NUM_param")),
                F.coalesce(F.col("CUAD_City"),      F.col("CUAD_City_param")),
                F.coalesce(F.col("CUAD_POSTCODE"),  F.col("cuad_postcode_param"))
            ).alias("FullAddress"),
            F.coalesce(F.col("CUAD_Country"), F.col("CUAD_Country_param")).alias("Country"),
            F.coalesce(F.col("CUAD_REGION"),  F.col("CUAD_REGION_param")).alias("County"),
            F.coalesce(F.col("CUAD_City"),    F.col("CUAD_City_param")).alias("City"),
            F.coalesce(F.col("CUAD_POSTCODE"),F.col("cuad_postcode_param")).alias("PostalCode"),
            F.when(
                ~F.coalesce(F.col("CUAD_STREET_3"), F.col("CUAD_STREET_3_param"), F.lit("N/A")).isin("N/A", "", "-"),
                F.coalesce(F.col("CUAD_STREET_3"), F.col("CUAD_STREET_3_param"))
            ).when(
                ~F.coalesce(F.col("CUAD_STREET_2"), F.col("CUAD_STREET_2_param"), F.lit("N/A")).isin("N/A", "", "-"),
                F.concat(
                    F.coalesce(F.col("CUAD_STREET_1"), F.col("CUAD_STREET_1_param")).cast("string"),
                    F.lit("-"),
                    F.coalesce(F.col("CUAD_STREET_2"), F.col("CUAD_STREET_2_param")).cast("string")
                )
            ).otherwise(F.coalesce(F.col("CUAD_STREET_1"), F.col("CUAD_STREET_1_param"))).alias("Street"),
            F.coalesce(F.col("CUAD_STREET_1"),  F.col("CUAD_STREET_1_param")).alias("Street1"),
            F.coalesce(F.col("CUAD_STREET_2"),  F.col("CUAD_STREET_2_param")).alias("Street2"),
            F.coalesce(F.col("CUAD_STREET_3"),  F.col("CUAD_STREET_3_param")).alias("Street3"),
            F.coalesce(F.col("CUAD_STREET_NUM"),F.col("CUAD_STREET_NUM_param")).alias("StreetNumber"),
            F.lit(None).alias("PhoneNumber1"),
            F.lit(None).alias("PhoneNumber2"),
            F.lit(None).alias("PhoneNumber3"),
            F.lit(None).alias("EmailAddress1"),
            F.lit(None).alias("EmailAddress2"),
            F.coalesce(F.col("CUAD_CONFIRMATION"),          F.col("CUAD_CONFIRMATION_param")).alias("IsValidFlag"),
            F.coalesce(F.col("CUAD_INTERNAL_CONFIRMATION"), F.col("CUAD_INTERNAL_CONFIRMATION_param")).cast("string").alias("CommunicationFlag"),
            F.lit(None).alias("IsPrimaryFlag"),
            F.lit(None).alias("CreatedBy"),
            F.coalesce(F.col("CUAD_CREATED"),               F.col("CUAD_CREATED_param")).alias("CreatedDate"),
            F.coalesce(F.col("CUAD_LAST_UPD"),              F.col("CUAD_LAST_UPD_param")).cast("date").alias("LastUpdatedDate"),
            F.lit(None).alias("OriginalTelephoneCode"),
            F.lit(None).alias("OriginalTelephoneType"),
            F.coalesce(F.col("CUAD_ACTIVE"),                F.col("CUAD_ACTIVE_param")).cast("string").alias("IsActiveFlag"),
            F.lit(None).alias("DoNotCallFlag"),
            F.lit(None).alias("IsDialerActivityFlag"),
            F.lit(None).alias("DialerActivityTelephoneNumber"),
            F.lit(None).alias("InternalConfirmationTypeCode"),
            F.lit(None).alias("InternalConfirmationType"),
            F.lit(None).alias("CollectionSourceTypeCode"),
            F.lit(None).alias("CollectionSourceType"),
            F.coalesce(F.col("CUAD_DO_NOT_SEND_FLAG"),      F.col("CUAD_DO_NOT_SEND_FLAG_param")).alias("DoNotSendFlag"),
            F.coalesce(F.col("CUAD_ADD_CONTACT_FLAG"),      F.col("CUAD_ADD_CONTACT_FLAG_param")).alias("AddContactFlag"),
            F.lit(None).alias("SerialNumber"),
            dim_customer_static.BusinessDate
        )

        # ================== Union ==================

        # Union the DataFrames
        contact_details_qca_df = contact_details_qca_df_address.union(
            contact_details_qca_df_phones
        )

        # ================== Append ==================

        contact_details_qca_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
  # type: ignore
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_contact_details_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)